# Reciprocal-based sigmoid Q4.4 test

Compare the existing TR-ln/TR-exp sigmoid with the new reciprocal-based sigmoid:

`E = exp(-abs(z))`, `denom = 255 + E`, `recip = reciprocal_u8(denom)`, then compute `(numer * recip + 128) >> 8`, with numerator `255` for `z >= 0` and `E` for `z < 0`.


In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f"Drive mount skipped: {exc}")

In [ ]:
import shutil
import sys
from pathlib import Path

import torch

PROJECT_DIR_PATH = globals().get("PROJECT_DIR_PATH", "/content/mrcp-tr-ptq")
PROJECT_DIR = Path(PROJECT_DIR_PATH).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

%cd {PROJECT_DIR_PATH}

GELU_EXT_DIR = PROJECT_DIR / "mrcp_quant" / "optimized_layers" / "gelu"
SIG_SCALE = 255

In [ ]:
# Rebuild after editing gelu_cuda*.{cpp,cu} or common/tr_math.cuh.
gelu_dir = Path(GELU_EXT_DIR)
shutil.rmtree(gelu_dir / "build", ignore_errors=True)
for so_path in gelu_dir.glob("_trptq_gelu*.so"):
    so_path.unlink()

!{sys.executable} -m pip install -v --no-build-isolation --no-cache-dir -e "{GELU_EXT_DIR}"

In [ ]:
import importlib
import _trptq_gelu
import trptq_gelu

trptq_gelu = importlib.reload(trptq_gelu)
print("Loaded:", trptq_gelu.__file__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())

if not hasattr(_trptq_gelu, "sigmoid_recip_arith"):
    raise RuntimeError("Missing sigmoid_recip_arith export. Rebuild and restart the runtime.")
print("sigmoid reciprocal export: OK")

In [ ]:
assert torch.cuda.is_available(), "Select a CUDA runtime first."
device = torch.device("cuda")
lut = trptq_gelu.build_exp_lut_u8(device=device)

x_q44 = torch.arange(-128, 128, device=device, dtype=torch.int16).to(torch.int8)
sig_old = trptq_gelu.sigmoid_u8_from_q44(x_q44, lut)
sig_recip = trptq_gelu.sigmoid_recip_u8_from_q44(x_q44, lut)
sig_float = torch.sigmoid(x_q44.float() / 16.0)
ref_u8 = torch.round(sig_float * SIG_SCALE).clamp(0, 255).to(torch.uint8)

def summarize(name, y):
    err = y.to(torch.int32).cpu() - ref_u8.to(torch.int32).cpu()
    return {
        "name": name,
        "max_q": int(err.abs().max()),
        "mean_q": err.abs().float().mean().item(),
        "rmse_f": (err.float() / SIG_SCALE).square().mean().sqrt().item(),
    }

rows = [summarize("old", sig_old), summarize("reciprocal", sig_recip)]
print(f"{'variant':<12} {'max_q':>8} {'mean_q':>10} {'rmse_f':>12}")
print("-" * 45)
for row in rows:
    print(f"{row['name']:<12} {row['max_q']:8d} {row['mean_q']:10.4f} {row['rmse_f']:12.4e}")

In [ ]:
x_cpu = x_q44.cpu().to(torch.int32)
old_cpu = sig_old.cpu().to(torch.int32)
recip_cpu = sig_recip.cpu().to(torch.int32)
ref_cpu = ref_u8.cpu().to(torch.int32)

print(f"{'z_q44':>6} {'z':>8} {'ref':>6} {'old':>6} {'recip':>6} {'old_err':>8} {'rec_err':>8}")
print("-" * 60)
for z in range(-128,127,1):
# for z in [-128, -96, -64, -32, -16, -8, 0, 8, 16, 32, 64, 96, 127]:
    i = int((x_cpu == z).nonzero(as_tuple=True)[0][0])
    print(
        f"{z:6d} {z / 16.0:8.4f} {int(ref_cpu[i]):6d} {int(old_cpu[i]):6d} {int(recip_cpu[i]):6d} "
        f"{int(old_cpu[i] - ref_cpu[i]):8d} {int(recip_cpu[i] - ref_cpu[i]):8d}"
    )

In [ ]:
def benchmark_cuda(fn, warmup=20, repeats=200):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(repeats):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeats


# Compare the old TR sigmoid kernel against float torch.sigmoid on the same Q4.4 input domain.
x_big_q44 = torch.randint(-128, 128, (1_000_000,), device=device, dtype=torch.int16).to(torch.int8)
old_ms = benchmark_cuda(lambda: trptq_gelu.sigmoid_u8_from_q44(x_big_q44, lut))

# Float baseline includes Q4.4 dequantization and quantization back to uint8, matching the kernel output format.
torch_float_ms = benchmark_cuda(
    lambda: torch.round(torch.sigmoid(x_big_q44.float() / 16.0) * SIG_SCALE).clamp(0, 255).to(torch.uint8)
)

old_big = trptq_gelu.sigmoid_u8_from_q44(x_big_q44, lut)
float_ref_big = torch.round(torch.sigmoid(x_big_q44.float() / 16.0) * SIG_SCALE).clamp(0, 255).to(torch.uint8)
err = old_big.to(torch.int32).cpu() - float_ref_big.to(torch.int32).cpu()

print("Old TR sigmoid vs float torch.sigmoid")
print(f"{'kernel':<20} {'ms':>10} {'speedup':>10}")
print("-" * 44)
print(f"{'old_tr_sigmoid':<20} {old_ms:10.4f} {torch_float_ms / old_ms:9.2f}x")
print(f"{'torch_float_sigmoid':<20} {torch_float_ms:10.4f} {'1.00':>9}x")
print()
print(f"max_abs_q={int(err.abs().max())}, mean_abs_q={err.abs().float().mean().item():.4f}, rmse_float={(err.float() / SIG_SCALE).square().mean().sqrt().item():.4e}")
